# WorkflowDirector — Colab Free T4 real-environment lab

**Target:** ComfyUI **v0.39.0**, WorkflowDirector, Colab Free, NVIDIA T4,
standard-memory runtime.

Order: clean smoke A→B, default-cache A-only, `--cache-none` A-only, then the
real GGUF/Power-LoRA A→B workflow. This notebook never calls Comfy `/free` or
`unload_all_models()`.

**Credentials:** no tokens are embedded. If this repository is private, create
a Colab Secret named `GITHUB_TOKEN`.


In [ ]:
# 0 — Runtime preflight
import os, sys, subprocess, json, pathlib, shutil

def sh(*args, check=True, capture=True, env=None):
    p = subprocess.run(list(args), check=check, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None, env=env)
    if capture and p.stdout:
        print(p.stdout)
    return p

print("Python:", sys.version)
sh("nvidia-smi")
gpu = subprocess.check_output(
    ["bash","-lc","nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
    text=True).strip()
print("GPU:", gpu)
if "T4" not in gpu:
    print("WARNING: smoke is allowed, but this is NOT a T4 acceptance run.")

import torch
PROTECTED = {"torch": torch.__version__, "cuda": torch.version.cuda}
print("Protected runtime:", PROTECTED)


In [ ]:
# 1 — Fresh ComfyUI v0.39.0 without replacing Colab PyTorch
COMFY_DIR = pathlib.Path("/content/ComfyUI")
COMFY_TAG = "v0.39.0"
if COMFY_DIR.exists():
    shutil.rmtree(COMFY_DIR)

sh("git","clone","--depth","1","--branch",COMFY_TAG,
   "https://github.com/Comfy-Org/ComfyUI.git",str(COMFY_DIR))

req = (COMFY_DIR/"requirements.txt").read_text().splitlines()
filtered = [x for x in req if x.strip() not in {"torch","torchvision"}]
req_path = pathlib.Path("/content/workflowdirector_requirements.txt")
req_path.write_text("\n".join(filtered)+"\n")
sh(sys.executable,"-m","pip","install","-q","-r",str(req_path),capture=False)

import torch
assert torch.__version__ == PROTECTED["torch"]
assert torch.version.cuda == PROTECTED["cuda"]
print("Comfy commit:", subprocess.check_output(
    ["git","-C",str(COMFY_DIR),"rev-parse","HEAD"], text=True).strip())
print("Torch preserved:", torch.__version__, torch.version.cuda)


In [ ]:
# 2 — Install WorkflowDirector from GitHub
WD_DIR = COMFY_DIR/"custom_nodes"/"ComfyUI-WorkflowDirector"
WD_REPO = "https://github.com/daneelolivaw2020/ComfyUI-WorkflowDirector.git"
if WD_DIR.exists():
    shutil.rmtree(WD_DIR)

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = os.environ.get("GITHUB_TOKEN")

env = os.environ.copy()
askpass = pathlib.Path("/content/git_askpass_workflowdirector.sh")
if token:
    env["WORKFLOWDIRECTOR_GITHUB_TOKEN"] = token
    askpass.write_text("""#!/bin/sh
case "$1" in
  *Username*) printf "%s\\n" "x-access-token" ;;
  *Password*) printf "%s\\n" "$WORKFLOWDIRECTOR_GITHUB_TOKEN" ;;
esac
""")
    askpass.chmod(0o700)
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

try:
    p = subprocess.run(
        ["git","clone","--depth","1","--branch","main",WD_REPO,str(WD_DIR)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    if p.returncode:
        raise RuntimeError("Clone failed. Add GITHUB_TOKEN to Colab Secrets if the repo is private.\n"+p.stdout[-1500:])
finally:
    askpass.unlink(missing_ok=True)
    env.pop("WORKFLOWDIRECTOR_GITHUB_TOKEN", None)

print("WorkflowDirector commit:", subprocess.check_output(
    ["git","-C",str(WD_DIR),"rev-parse","HEAD"], text=True).strip())


In [ ]:
# 3 — Prepare smoke workflow files
LAB_FILES = pathlib.Path("/content/workflowdirector_lab")
LAB_FILES.mkdir(exist_ok=True)
for name in ["lab_smoke_A.json","lab_smoke_B.json"]:
    shutil.copy2(WD_DIR/"workflows"/name, LAB_FILES/name)
    print(LAB_FILES/name)


In [ ]:
# 4 — Start/restart ComfyUI
# First run False. For the controlled comparison later change ONLY this to True.
import signal, time, requests
PORT = 8188
CACHE_NONE = False
OBSERVATION_SECONDS = 1.0
LOG = pathlib.Path("/content/comfyui_workflowdirector.log")
PIDFILE = pathlib.Path("/content/comfyui_workflowdirector.pid")

def stop_comfy():
    if PIDFILE.exists():
        try:
            pid = int(PIDFILE.read_text())
            os.kill(pid, signal.SIGTERM)
            for _ in range(40):
                try: os.kill(pid, 0)
                except ProcessLookupError: break
                time.sleep(0.25)
            else: os.kill(pid, signal.SIGKILL)
        except (ValueError, ProcessLookupError):
            pass
        PIDFILE.unlink(missing_ok=True)

stop_comfy()
cmd = [sys.executable,"main.py","--listen","0.0.0.0","--port",str(PORT)]
if CACHE_NONE:
    cmd.append("--cache-none")
env = os.environ.copy()
env["WORKFLOWDIRECTOR_OBSERVATION_SECONDS"] = str(OBSERVATION_SECONDS)
log_handle = open(LOG,"w",buffering=1)
proc = subprocess.Popen(cmd,cwd=str(COMFY_DIR),env=env,
    stdout=log_handle,stderr=subprocess.STDOUT)
PIDFILE.write_text(str(proc.pid))

base = f"http://127.0.0.1:{PORT}"
for _ in range(240):
    if proc.poll() is not None:
        raise RuntimeError("Comfy exited early.\n"+LOG.read_text(errors="replace")[-5000:])
    try:
        r = requests.get(base+"/workflowdirector/health",timeout=1)
        if r.ok:
            print(json.dumps(r.json(),indent=2))
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    raise TimeoutError("Comfy did not become ready.\n"+LOG.read_text(errors="replace")[-5000:])
print("Mode:", "CACHE_NONE" if CACHE_NONE else "DEFAULT CACHE")


In [ ]:
# 5 — Open ComfyUI
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=900)


## 6 — Smoke A → B

In ComfyUI:

1. turn **Auto Queue off**;
2. open `lab_smoke_A.json`;
3. open bottom panel **Workflow Director Lab**;
4. **Capture current as A**;
5. open `lab_smoke_B.json`;
6. **Capture current as B**;
7. click **Run A → B**;
8. queue nothing else while it runs.

Pass: phase `completed`, two different native job IDs, baseline and post-job
memory rows for both steps, no queue-interference error, kernel alive.

If smoke fails, stop. Do not install GGUF yet.


In [ ]:
# 7 — Download smoke workflows to the browser if needed
from google.colab import files
for name in ["lab_smoke_A.json","lab_smoke_B.json"]:
    files.download(str(LAB_FILES/name))


In [ ]:
# 8 — Read-only diagnostics
import requests, json
for endpoint in [
    "/workflowdirector/health",
    "/workflowdirector/memory",
    "/api/jobs?status=pending,in_progress",
]:
    r = requests.get(base+endpoint,timeout=5)
    print("\n==",endpoint,r.status_code,"==")
    print(r.text[:12000])
print("\n== log tail ==")
print(LOG.read_text(errors="replace")[-8000:])


## 9 — Controlled cache experiment

After smoke passes, use the exact same known-working heavy Workflow A.

1. With `CACHE_NONE=False`, capture A and run **A only**. Record the table.
2. Return to cell 4, change **only** `CACHE_NONE=True`, restart Comfy and repeat
   the exact same A-only test.

Do not simultaneously change quantization, model paths, LoRA stack,
pinned-memory flags, DynamicVRAM, prompt, seed, or workflow structure.


In [ ]:
# 10 — OPTIONAL heavy custom nodes, only after smoke passes
HEAVY_NODES = {
    "ComfyUI-GGUF": (
        "https://github.com/city96/ComfyUI-GGUF.git",
        "6ea2651e7df66d7585f6ffee804b20e92fb38b8a",
    ),
    "rgthree-comfy": (
        "https://github.com/rgthree/rgthree-comfy.git",
        "d7297b42f0fa78bec1c0411478f887423c249502",
    ),
}
stop_comfy()
for name,(repo,ref) in HEAVY_NODES.items():
    dst = COMFY_DIR/"custom_nodes"/name
    if dst.exists(): shutil.rmtree(dst)
    sh("git","clone",repo,str(dst))
    sh("git","-C",str(dst),"checkout","--detach",ref)
    req = dst/"requirements.txt"
    if req.exists():
        sh(sys.executable,"-m","pip","install","-q","-r",str(req),capture=False)
    print(name, subprocess.check_output(
        ["git","-C",str(dst),"rev-parse","HEAD"],text=True).strip())
import torch
assert torch.__version__ == PROTECTED["torch"]
assert torch.version.cuda == PROTECTED["cuda"]
print("Restart Comfy with cell 4.")


## 11 — Real GGUF / Power-LoRA A → B

Use the **actual workflows that already work individually**.

- A: first Klein/Qwen/VAE generation + its LoRA stack.
- B: second independent generation + its LoRA stack.
- no unload node;
- no `/free`;
- fixed seeds;
- Auto Queue off.

If needed, run each normally once to verify it still works on v0.39.0, then
restart Comfy before measurement. Capture A and B in the Lab and run A → B.

The first success criterion is not “VRAM goes to zero.” It is that A completes
as one native job, its boundary runs, B begins as a new native job, B completes,
and Colab/T4 remains alive without dangerous system-RAM exhaustion.

If normal Queue works but the captured copy fails around a dynamic custom node,
preserve the log: that points to the current `graphToPrompt()` capture
limitation rather than automatically to the memory boundary.


In [ ]:
# 12 — Export an evidence bundle
import datetime, zipfile
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
root = pathlib.Path(f"/content/workflowdirector_evidence_{stamp}")
root.mkdir()
def write(name, content):
    (root/name).write_text(content if isinstance(content,str) else json.dumps(content,indent=2))
for endpoint,fname in [
    ("/workflowdirector/health","health.json"),
    ("/workflowdirector/memory","memory.json"),
    ("/api/jobs?status=pending,in_progress","active_jobs.json"),
]:
    try: write(fname, requests.get(base+endpoint,timeout=5).text)
    except Exception as e: write(fname,repr(e))
write("comfy_commit.txt",subprocess.check_output(["git","-C",str(COMFY_DIR),"rev-parse","HEAD"],text=True))
write("workflowdirector_commit.txt",subprocess.check_output(["git","-C",str(WD_DIR),"rev-parse","HEAD"],text=True))
write("nvidia_smi.txt",subprocess.check_output(["nvidia-smi"],text=True))
write("python.txt",sys.version)
write("torch.txt",f"{torch.__version__}\nCUDA={torch.version.cuda}\n")
write("comfy_log_tail.txt",LOG.read_text(errors="replace")[-50000:])
zip_path = pathlib.Path(str(root)+".zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in root.iterdir(): z.write(p,arcname=p.name)
print(zip_path)
from google.colab import files
files.download(str(zip_path))
